# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/begs44/flyrank-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

The FlyRank paper is written for a broad audience and is open about its own limits: ML is labelled "exploratory", and no p-values are claimed. Below are the questions I would ask as a reviewer. I also try each one on my own data in the code cell.

**Finding A: "Logistic regression (71% holdout accuracy) describing which features separate growing from declining pages" (ML appendix).**
- *Where does the label come from?* Growing vs declining comes from the 30-day vs previous-30-day impression change, with a ±10% threshold. Several features (impressions, days visible) are 90-day totals that contain those same 30 days. How much of the 71% comes from features that already see the label window?
- *Does the validation support the claim?* The split is a random 80/20, so pages from the same brand can sit in both train and test. Would 71% hold up on brands the model has never seen?
- *What is 71% compared to?* No base rate is given. If one class makes up 62% of pages, 71% is 9 points of skill, not 71.
- **On my data:** my model gets 0.70 accuracy under a random split (very close to the paper's number). Under a client-grouped split it drops to 0.645, against a 0.542 floor. Section 2 shows this.

**Finding B: "Refreshing mature pages: 3.2× health boost and 57× more impressions" (Finding #4).**
- *Where does the comparison come from?* It compares old pages that were refreshed with old pages that were not. Refreshing was not random: editors may pick pages that already have demand. Are the two groups comparable before the refresh?
- *Does the design support the wording?* "Refresh produces 57× impressions" reads as cause and effect. A before/after comparison on the same pages, or a matched control group, would make it a lot stronger.
- **On my data:** recently updated pages differ from the rest *before* any credit is given (median impressions 474 vs 1,598, median age 187 vs 258 days). In 19 of 32 clients, more than 90% of the pages fall into the same update group. So "when was it updated" is mostly "which client is it", and the comparison picks up client differences.

In [3]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score

SEED = 42
url = "https://raw.githubusercontent.com/begs44/flyrank-ml/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df["target_decay"] = (df["trend_direction"] == "down").astype(int)
base = df["target_decay"].mean()

# ---- Same Week-5 feature set ----
df["avg_position"] = df["avg_position"].replace(0, np.nan)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
for c in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{c}"] = np.log1p(df[c])
NUMERIC = ["search_volume", "competition", "cpc", "word_count",
           "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
           "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
           "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
           "has_word_count", "has_keyword_data"]
CATEGORICAL = ["content_type", "main_intent", "competition_level"]
df[CATEGORICAL] = df[CATEGORICAL].fillna("unknown")
y = df["target_decay"]

# ---- Paper question 1, tried on my data: what does "accuracy" mean without a base rate? ----
print(f"Base rate (share declining): {base:.3f}")
print(f"Accuracy of a model that always says 'declining': {max(base, 1 - base):.3f}  <- the floor any accuracy must beat\n")

# ---- Paper question 2, tried on my data: are recently-updated pages comparable to the others? ----
grp = np.where(df["days_since_last_update"] < 60, "updated <60d ago", "updated 60d+ ago")
comp = df.groupby(grp).agg(pages=("target_decay", "size"),
                           median_impressions=("impressions_90d", "median"),
                           median_age_days=("content_age_days", "median"),
                           clients=("client_id", "nunique"),
                           decline_rate=("target_decay", "mean")).round(3)
print("Recently updated vs not — do the groups differ BEFORE we credit the update?")
print(comp.to_string())
top_client_share = pd.Series(grp).groupby(df["client_id"].values).agg(lambda s: (s == "updated 60d+ ago").mean())
print(f"\nClients where >90% of pages fall in one update group: {(top_client_share > 0.9).sum() + (top_client_share < 0.1).sum()} of {df['client_id'].nunique()}")
print("=> 'when a page was updated' is mostly 'which client it belongs to' — a comparison, not a controlled test.")

Base rate (share declining): 0.542
Accuracy of a model that always says 'declining': 0.542  <- the floor any accuracy must beat

Recently updated vs not — do the groups differ BEFORE we credit the update?
                  pages  median_impressions  median_age_days  clients  decline_rate
updated 60d+ ago   9397              1598.0            258.0       24         0.608
updated <60d ago  20603               474.0            187.0       31         0.512

Clients where >90% of pages fall in one update group: 19 of 32
=> 'when a page was updated' is mostly 'which client it belongs to' — a comparison, not a controlled test.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Before → after:** I re-ran my Week-5 models under two designs and kept everything else identical: same features, same models, same seed, out-of-fold scores.
- **Before: random 5-fold.** Pages from the same client land in both train and test.
- **After: grouped 5-fold by `client_id`.** Every client is tested only on a model that never saw it.

| Model | P@50 random | P@50 grouped | gap | ROC-AUC random | ROC-AUC grouped |
|---|---|---|---|---|---|
| Week-4 rule baseline | 0.704 | 0.752 | −0.048 | 0.612 | 0.632 |
| Logistic Regression | 0.864 | 0.800 | +0.064 | 0.710 | 0.661 |
| Gradient Boosting | **0.944** | **0.808** | **+0.136** | 0.777 | 0.678 |

- **The random split inflates the learned models,** most of all Gradient Boosting (−0.136 at P@50, −0.099 at ROC-AUC). The flexible model memorizes each client's habits. The rule has nothing to memorize, so it does not gain from the random split.
- **The gap between learned models and the rule shrinks** from about 0.24 to about 0.06 under the honest split. **The grouped numbers are the ones I report.**
- **Accuracy at 0.5** is 0.702 under a random split and 0.645 under a grouped split, against a 0.542 floor.

**Real failure examples (grouped split):**
- **False alarms:** three pages scored 0.95+ but were not declining. All three sit at position ~2, have 2,000–4,800 impressions with only 3 clicks, are about 140 days old and were updated 104 days ago. They look exactly like decaying pages. The label only compares two 30-day windows, and these pages simply held steady.
- **Missed declines:** three pages scored ≤ 0.06 but were labelled "declining". Each had **1 impression in 90 days.** Going from 1 impression to 0 counts as "down", so this is **label noise,** not a modelling failure. Section 3 measures how common it is.

In [4]:
def prep(scale, num_cols=None):
    num_cols = num_cols or NUMERIC
    num = make_pipeline(SimpleImputer(strategy="median"), StandardScaler()) if scale else SimpleImputer(strategy="median")
    return make_column_transformer((num, num_cols), (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL))

def make_lr(num_cols=None):
    return make_pipeline(prep(True, num_cols), LogisticRegression(max_iter=2000, C=0.5))

def make_gb(num_cols=None):
    return make_pipeline(prep(False, num_cols), HistGradientBoostingClassifier(
        max_depth=4, learning_rate=0.05, max_iter=300, min_samples_leaf=100, random_state=SEED))

def baseline_rule_score(train, test):
    """Week-4 rule, unchanged. Position-band CTR medians from TRAIN only."""
    bands = [0, 3, 5, 10, 15, 20]
    vis = lambda d: (d["impressions_90d"] >= 500) & (d["avg_position"] > 0) & (d["avg_position"] <= 20)
    exp = train[vis(train)].groupby(pd.cut(train.loc[vis(train), "avg_position"], bands), observed=True)["ctr"].median()
    e = pd.cut(test["avg_position"], bands).map(exp).astype(float)
    shortfall = (1 - test["ctr"] / e).clip(0, 1)
    return np.where(vis(test), shortfall * np.log10(test["impressions_90d"]), 0.0) + 1e-9 * test["impressions_90d"].values

def p_at_k(y_true, s, k):
    return float(np.asarray(y_true)[np.argsort(-np.asarray(s), kind="stable")[:k]].mean())

def evaluate(folds, num_cols=None, extra=None, models=("rule", "lr", "gb")):
    """Out-of-fold scores for every model; returns mean metrics and OOF GB scores."""
    num_cols = num_cols or NUMERIC
    cols = num_cols + CATEGORICAL
    rows, oof = [], pd.Series(np.nan, index=df.index)
    for tr, te in folds:
        Xtr, Xte = df.iloc[tr][cols], df.iloc[te][cols]
        if extra is not None:
            Xtr, Xte = Xtr.assign(extra=extra.iloc[tr].values), Xte.assign(extra=extra.iloc[te].values)
        scores = {}
        if "rule" in models: scores["Week-4 rule baseline"] = baseline_rule_score(df.iloc[tr], df.iloc[te])
        if "lr" in models:   scores["Logistic Regression"] = make_lr(num_cols + (["extra"] if extra is not None else [])).fit(Xtr, y.iloc[tr]).predict_proba(Xte)[:, 1]
        if "gb" in models:
            scores["Gradient Boosting"] = make_gb(num_cols + (["extra"] if extra is not None else [])).fit(Xtr, y.iloc[tr]).predict_proba(Xte)[:, 1]
            oof.iloc[te] = scores["Gradient Boosting"]
        for name, s in scores.items():
            rows.append({"model": name, "P@50": p_at_k(y.iloc[te], s, 50), "ROC-AUC": roc_auc_score(y.iloc[te], s),
                         "accuracy@0.5": accuracy_score(y.iloc[te], np.asarray(s) >= 0.5) if name != "Week-4 rule baseline" else np.nan})
    return pd.DataFrame(rows).groupby("model", sort=False).mean().round(3), oof

random_folds  = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(df, y))
grouped_folds = list(GroupKFold(n_splits=5).split(df, y, groups=df["client_id"]))

before, _ = evaluate(random_folds)
after, oof_gb = evaluate(grouped_folds)
ba = before.add_suffix(" | random").join(after.add_suffix(" | grouped"))
ba["P@50 gap"] = (ba["P@50 | random"] - ba["P@50 | grouped"]).round(3)
ba["ROC-AUC gap"] = (ba["ROC-AUC | random"] - ba["ROC-AUC | grouped"]).round(3)
print(f"BEFORE = random 5-fold (clients mixed) | AFTER = grouped 5-fold by client | base rate {base:.3f}\n")
print(ba[["P@50 | random", "P@50 | grouped", "P@50 gap", "ROC-AUC | random", "ROC-AUC | grouped", "ROC-AUC gap",
          "accuracy@0.5 | random", "accuracy@0.5 | grouped"]].to_string(), "\n")

# ---- Real failure examples (grouped split, out-of-fold Gradient Boosting scores) ----
show = ["impressions_90d", "clicks_90d", "ctr", "avg_position", "content_age_days",
        "days_since_last_update", "days_with_impressions", "main_intent"]
df["oof_gb"] = oof_gb
print("False alarms — highest scores, NOT declining:")
print(df[y == 0].nlargest(3, "oof_gb")[["oof_gb"] + show].round(3).to_string(index=False), "\n")
print("Missed declines — lowest scores, but DID decline:")
print(df[y == 1].nsmallest(3, "oof_gb")[["oof_gb"] + show].round(3).to_string(index=False))

BEFORE = random 5-fold (clients mixed) | AFTER = grouped 5-fold by client | base rate 0.542

                      P@50 | random  P@50 | grouped  P@50 gap  ROC-AUC | random  ROC-AUC | grouped  ROC-AUC gap  accuracy@0.5 | random  accuracy@0.5 | grouped
model                                                                                                                                                         
Week-4 rule baseline          0.704           0.752    -0.048             0.612              0.632       -0.020                    NaN                     NaN
Logistic Regression           0.864           0.800     0.064             0.710              0.661        0.049                  0.658                   0.622
Gradient Boosting             0.944           0.808     0.136             0.777              0.678        0.099                  0.702                   0.645 

False alarms — highest scores, NOT declining:
 oof_gb  impressions_90d  clicks_90d  ctr  avg_position  content

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Leakage audit on my final feature set:**

- **(a) Label-derived columns and product flags: none.** `trend_direction`, `trend_pct`, all `*_last_30d` / `*_prev_30d` columns, IDs, and FlyRank's pre-made tier buckets are kept out, and the code checks this. My Week-4 rule score is used only as a baseline to beat, never as a feature.
- **(b) No single feature is "too good".** The strongest single feature reaches ROC-AUC 0.59 on its own (`content_age_days`). That is nowhere near the > 0.80 that would suggest a hidden answer.
- **(c) The test harness works.** When I add `trend_pct` (the label's source) on purpose, Gradient Boosting jumps to P@50 = 1.0 and ROC-AUC = 1.0. So the setup would catch a leak like this. It is removed again.
- **(d) Overlapping windows: the most important finding of this audit.** The label compares the last 30 days with the 30 days before. My traffic features are 90-day totals that **contain** both of those windows.
  - With static metadata only (age, staleness, word count, keyword data), the model reaches P@50 = 0.496, which is **below the 0.542 base rate**.
  - Removing just the day-count columns barely matters (0.796).
  - So all of the model's skill, and my Week-4 rule's skill too, comes from traffic windows that overlap the label. **The model recognizes decline that is already in progress more than it forecasts decline before it starts.** This is not a bug in the comparison (the rule and the model use the same kind of inputs), but it limits what I can claim.
  - The honest fix is a **time-aware design** in the warehouse data: features only from days 61–150 back, label from the last 30 days. I name this as the next step.
- **(e) Label noise.** 3,746 pages had fewer than 10 impressions in 90 days, and 753 of them (20%) are still labelled "declining". For these pages, one impression decides the label. These are the "missed declines" from section 2.

In [5]:
# ---- (a) Name check: no label, label source, 30-day trend window, ID or product flag in the features ----
FORBIDDEN = {"trend_direction", "trend_pct", "target_decay", "is_declining_label", "content_id", "client_id",
             "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
             "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
             "impression_tier", "position_tier", "age_tier", "freshness_tier"}   # tiers = existing-system buckets
hits = FORBIDDEN & set(NUMERIC + CATEGORICAL)
print("(a) Forbidden columns in feature set:", hits or "none")

# ---- (b) Single-feature scan: any one column that separates the label 'too well' is a suspect ----
scan = {c: roc_auc_score(y, df[c].fillna(df[c].median())) for c in NUMERIC}
scan = pd.Series(scan).apply(lambda a: max(a, 1 - a)).sort_values(ascending=False).round(3)
print("\n(b) Single-feature ROC-AUC (direction-free), top 5 — anything > 0.80 would be a red flag:")
print(scan.head(5).to_string())

# ---- (c) Harness test: inject the label source on purpose. The score MUST jump, or the test is broken ----
leaky, _ = evaluate(grouped_folds, extra=df["trend_pct"].fillna(0), models=("gb",))
print(f"\n(c) Gradient Boosting + trend_pct (deliberate leak): P@50 = {leaky.loc['Gradient Boosting', 'P@50']}, "
      f"ROC-AUC = {leaky.loc['Gradient Boosting', 'ROC-AUC']}   vs honest {after.loc['Gradient Boosting', 'P@50']} / {after.loc['Gradient Boosting', 'ROC-AUC']}")
print("    -> the harness catches leakage; trend_pct stays OUT.")

# ---- (d) Window overlap: 90-day totals CONTAIN the 30-day label window. How much skill depends on them? ----
STATIC = ["search_volume", "competition", "cpc", "word_count", "content_age_days",
          "days_since_last_update", "has_word_count", "has_keyword_data"]          # metadata, no traffic window
WINDOWED = [c for c in NUMERIC if c not in STATIC]
static_only, _ = evaluate(grouped_folds, num_cols=STATIC, models=("lr", "gb"))
no_daycount, _ = evaluate(grouped_folds, num_cols=[c for c in NUMERIC if not c.startswith("days_with")], models=("gb",))
print("\n(d) Window-overlap check (grouped split, Gradient Boosting):")
wtab = pd.DataFrame({
    "features": ["all Week-5 features", "without day-count columns", "static metadata only (no traffic windows)", "Week-4 rule baseline"],
    "P@50": [after.loc["Gradient Boosting", "P@50"], no_daycount.loc["Gradient Boosting", "P@50"],
             static_only.loc["Gradient Boosting", "P@50"], after.loc["Week-4 rule baseline", "P@50"]],
    "ROC-AUC": [after.loc["Gradient Boosting", "ROC-AUC"], no_daycount.loc["Gradient Boosting", "ROC-AUC"],
                static_only.loc["Gradient Boosting", "ROC-AUC"], after.loc["Week-4 rule baseline", "ROC-AUC"]]})
print(wtab.to_string(index=False))
print(f"\nWindowed (overlapping) features: {WINDOWED}")

# ---- (e) Label noise: how many 'declines' are tiny pages where one impression decides the label? ----
tiny = df["impressions_90d"] < 10
print(f"\n(e) Pages with <10 impressions in 90 days: {tiny.sum():,} — of these, labelled declining: {y[tiny].sum():,} "
      f"({y[tiny].mean():.3f})")

(a) Forbidden columns in feature set: none

(b) Single-feature ROC-AUC (direction-free), top 5 — anything > 0.80 would be a red flag:
content_age_days         0.591
log_impressions_90d      0.584
days_with_impressions    0.579
search_volume            0.560
has_keyword_data         0.540

(c) Gradient Boosting + trend_pct (deliberate leak): P@50 = 1.0, ROC-AUC = 1.0   vs honest 0.808 / 0.678
    -> the harness catches leakage; trend_pct stays OUT.

(d) Window-overlap check (grouped split, Gradient Boosting):
                                 features  P@50  ROC-AUC
                      all Week-5 features 0.808    0.678
                without day-count columns 0.796    0.657
static metadata only (no traffic windows) 0.496    0.541
                     Week-4 rule baseline 0.752    0.632

Windowed (overlapping) features: ['log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'ctr', 'avg_position', 'engagement_

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest sentences, rewritten to match the evidence** (the code below prints them with the live numbers):

1. **W1 — "A learned ranking model surfaces genuine decay far more effectively than basic heuristics."**
   → Under a client-grouped split, the model's top-50 had a measured decline rate of 0.81, against 0.75 for my rule (base rate 0.54). That is a modest, directional gain, not "far more".
2. **W1 — "Our model achieved Precision@50 = 0.720."**
   → That number came from a starter notebook with a different split. I report my own grouped 5-fold result, with its fold spread.
3. **W3 — "search_volume is excluded due to proven lack of correlation."**
   → search_volume showed a weak observed link to *impressions* (r = 0.001). On its own, though, it separates the decline label about as well as other features (single-feature AUC 0.56). Weak for traffic is not the same as useless for this label, and "proven" was too strong.
4. **W1 — "This prevents traffic loss and maximizes editorial efficiency."**
   → The ranking is a decision-support aid for choosing which pages to review first. It does not show that a refresh causes recovery.
5. **W5 — "The model predicts which pages will decline."**
   → The model ranks pages that are observed to be declining in the latest 30-day window, using 90-day signals that overlap that window. It describes decline in progress more than it forecasts it.

**My headline claim from now on:** *On 30k pages from 32 clients, with a client-grouped split, a gradient-boosting ranking put a measured 81% declining pages in its top 50, against 75% for a transparent CTR rule and a 54% base rate. The signals overlap the label window, so this is decision support for triage, not a forecast.*

In [6]:
claims = pd.DataFrame([
    ("W1", "A learned ranking model ... surfaces genuine decay far more effectively than basic heuristics.",
     f"Under a client-grouped split, the model's top-50 had a measured decline rate of {after.loc['Gradient Boosting', 'P@50']:.2f} "
     f"vs {after.loc['Week-4 rule baseline', 'P@50']:.2f} for my rule (base rate {base:.2f}) — a modest, directional gain."),
    ("W1", "Our model achieved Precision@50 = 0.720.",
     "That number came from a starter notebook with a different split; my own grouped 5-fold result is the one I report, with its fold spread."),
    ("W3", "search_volume is excluded due to proven lack of correlation (r = 0.001).",
     f"search_volume showed a weak observed link to impressions (r = 0.001), but on its own it separates the decline label "
     f"about as well as other features (single-feature AUC {scan['search_volume']:.2f}); weak for traffic, not useless for this label."),
    ("W1", "This prevents traffic loss and maximizes editorial efficiency.",
     "The ranking is a decision-support aid for choosing which pages to review first; it does not show that a refresh causes recovery."),
    ("W5", "The model predicts which pages will decline.",
     "The model ranks pages that are observed to be declining in the latest 30-day window, using 90-day signals that overlap that window — "
     "it describes decline in progress more than it forecasts it."),
], columns=["week", "original claim", "rewritten (observed / measured / directional / decision-support)"])
pd.set_option("display.max_colwidth", None)
for _, r in claims.iterrows():
    print(f"[{r['week']}] BEFORE: {r['original claim']}\n      AFTER:  {r.iloc[2]}\n")

# Numbers behind the rewritten claims, in one place
print(f"Grouped-split P@50  — model {after.loc['Gradient Boosting', 'P@50']} | LR {after.loc['Logistic Regression', 'P@50']} | "
      f"rule {after.loc['Week-4 rule baseline', 'P@50']} | base rate {base:.3f}")
print(f"Random-split P@50 for the same model: {before.loc['Gradient Boosting', 'P@50']} (inflated by client memorization)")
print(f"Static-only model P@50: {static_only.loc['Gradient Boosting', 'P@50']} (skill depends on overlapping traffic windows)")

[W1] BEFORE: A learned ranking model ... surfaces genuine decay far more effectively than basic heuristics.
      AFTER:  Under a client-grouped split, the model's top-50 had a measured decline rate of 0.81 vs 0.75 for my rule (base rate 0.54) — a modest, directional gain.

[W1] BEFORE: Our model achieved Precision@50 = 0.720.
      AFTER:  That number came from a starter notebook with a different split; my own grouped 5-fold result is the one I report, with its fold spread.

[W3] BEFORE: search_volume is excluded due to proven lack of correlation (r = 0.001).
      AFTER:  search_volume showed a weak observed link to impressions (r = 0.001), but on its own it separates the decline label about as well as other features (single-feature AUC 0.56); weak for traffic, not useless for this label.

[W1] BEFORE: This prevents traffic loss and maximizes editorial efficiency.
      AFTER:  The ranking is a decision-support aid for choosing which pages to review first; it does not show that a ref

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.